# QuantumLab: GPU validation on a real NVIDIA GPU

This notebook checks the CuPy backend of `SplitOperatorSolver` on real CUDA hardware:

1. it runs the full test suite, including the **real-CuPy tests**, which are skipped on GPU-less CI runners;
2. it benchmarks CPU vs GPU on 3D grids and checks the GPU result against the CPU result;
3. it prints a summary you can paste back.

**Before running:** *Runtime → Change runtime type → Hardware accelerator: GPU* (a T4 is enough), then *Runtime → Run all*. It takes about 5 minutes.

In [ ]:
# 1. GPU visible?
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2. Get QuantumLab and install it with its test dependencies
import os, subprocess, sys
if os.path.basename(os.getcwd()) == 'quantum-simulation':   # safe to run this cell again
    os.chdir('..')
REPO = 'https://github.com/rasidi3112/quantum-simulation.git'
if not os.path.isdir('quantum-simulation'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', './quantum-simulation[dev]'], check=True)

# CuPy ships with Colab's GPU runtime; install it only if it is missing and an
# NVIDIA driver is present (without one the GPU parts below are skipped)
import shutil
try:
    import cupy
except ImportError:
    if shutil.which('nvidia-smi'):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'cupy-cuda12x'], check=True)
        import cupy
    else:
        cupy = None
        print('No NVIDIA driver found: CuPy not installed, GPU parts will be skipped.')
os.chdir('quantum-simulation')
print('working directory:', os.getcwd())

In [ ]:
# 3. Environment report
import platform
import numpy, scipy
import quantumlab
from quantumlab.solvers.backends import cupy_available, get_backend

gpu = cupy_available()
info = {
    'python': platform.python_version(),
    'quantumlab': quantumlab.__version__,
    'numpy': numpy.__version__,
    'scipy': scipy.__version__,
    'cupy': cupy.__version__ if cupy is not None else 'not installed',
    'cuda_device': cupy.cuda.runtime.getDeviceProperties(0)['name'].decode() if gpu else 'none',
    'auto_backend': get_backend('auto').name,
}
for k, v in info.items():
    print(f'{k:>13}: {v}')
if not gpu:
    print('\nNo CUDA device found: switch the runtime to GPU and run all cells again.')

In [ ]:
# 4. Full test suite. With a GPU, the real-CuPy tests in tests/test_backends.py
#    and the legacy tests/test_3d_gpu_solver.py run instead of being skipped.
import subprocess, sys, os
env = dict(os.environ, MPLBACKEND='Agg')
res = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-rs', '--color=no', '-p', 'no:cacheprovider'],
                     capture_output=True, text=True, env=env)
print(res.stdout[-4000:])
print(res.stderr[-2000:])
pytest_summary = res.stdout.strip().splitlines()[-1] if res.stdout.strip() else 'no output'

In [ ]:
# 5. Real-CuPy tests on their own, verbose (they compare GPU against CPU results)
res_gpu = subprocess.run([sys.executable, '-m', 'pytest', '-v', '--color=no', '-p', 'no:cacheprovider',
                          'tests/test_backends.py::test_real_cupy_matches_cpu', 'tests/test_3d_gpu_solver.py'],
                         capture_output=True, text=True, env=env)
print(res_gpu.stdout[-4000:])
gpu_test_summary = res_gpu.stdout.strip().splitlines()[-1] if res_gpu.stdout.strip() else 'no output'

In [ ]:
# 6. CPU vs GPU benchmark: 3D rotating Gross–Pitaevskii, 50 steps per grid size.
#    The last column is the largest |ψ_GPU − ψ_CPU| after those steps.
bench = subprocess.run([sys.executable, 'benchmarks/gpu_split_operator.py', '--sizes', '64', '96', '128', '160',
                        '--steps', '50'], capture_output=True, text=True, env=env)
print(bench.stdout)
print(bench.stderr[-2000:])

In [ ]:
# 7. Summary: copy everything printed below and paste it back
print('```')
for k, v in info.items():
    print(f'{k}: {v}')
print(f'full test suite: {pytest_summary}')
print(f'real-GPU tests: {gpu_test_summary}')
print()
print(bench.stdout.strip())
print('```')